In [ ]:
# Spark SQL

1. part of Apache spark that works with structured data using sql queries or programming api

Functionalities of spark sql

1. run sql queries on big data
2. Dataframe api - easy to use in languages like python, java, scala
3. support multiple data sources
        - csv, json, parquet files, databases, big data storage system

4. integration with hive (use existing hive tables and run them)
5. optimization (fast processing) - catalyst optimizer
6. support structured and semistructed
7. temporary views

In [ ]:
Key features of spark sql

1. sql support

2. dataframe and dataset api

3. high performance

4. schema inference and management

5. lazy evaluated

In [ ]:
df.createOrReplaceTempView("employees") # creation of temporary virtual table

In [ ]:
- logical plan is already defined not fully executed
- data may be partitioned

1. Register employees in the catalog (metadata layer)

2. associate the name with the logical plan of dataframe

3. if it already exists - replace it (No data movement, no repartitioning, no computation)

4. Even after creating the view
           - spark still has not touched the actual data, no partitions are processed

5. spark.sql("Select * from employees where")

          - reads from registered view (employees)
          - apply filters, projections

6. catalyst optimizer kicks in
         - pushing filter down 
         - remove unnecessary column
        - reorders operations

7. physical plan is created
    - spark will decide how many partitions to use, which ops will go in parallel, whether to shuffle data

8. Execution + partition processing
    - data (partitions), each partition is processed by task, task will run in parallel across executions

9. shuffling (if needed)
   query - group by ,join, order by (data is reshuflled across partitions, redistribution happens bcoz of keys)

In [ ]:
spark sql vs traditional sql

Traditional sql
- mysql, postgres
- structured, real time transactions
- CRUD
- faster for small queries, ACID strong compliance, OLTP systems

spark sql 
- analytical processing
- batch processing
- complex transformations (OLAP)




In [ ]:
when to use spark sql api?

1. business analysts - sql
2. hive (direct sql queries)
3. quick exploration

when to use dataframe api

1. complex transformation - multi steps, conditions, loops
2. production pipelines ETL
3. type safety (scala/java)
4. dynamic logic (programmatic control)

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL").getOrCreate()

In [2]:
df = spark.read.csv("data/employee_data (1).csv", header = True, inferSchema = True)
df.show()

+-------+---+------+----------+
|   name|age|salary|department|
+-------+---+------+----------+
|  Alice| 25|  5000|        HR|
|    Bob| 30|  6000|        IT|
|Charlie| 35|  7000|   Finance|
|  David| 28|  5500|     Sales|
|    Eve| 32|  6200|        IT|
|  Frank| 40|  7500|   Finance|
|  Grace| 27|  5300|        HR|
|   Hank| 33|  6400|        IT|
|    Ivy| 29|  5700|     Sales|
|   Jack| 38|  7200|   Finance|
|  Kelly| 26|  5100|        HR|
|   Liam| 31|  5900|        IT|
|    Mia| 34|  6800|   Finance|
|   Noah| 27|  5400|     Sales|
| Olivia| 36|  7100|        IT|
|  Peter| 29|  5600|        HR|
|  Quinn| 37|  7300|   Finance|
|   Rose| 28|  5500|     Sales|
|    Sam| 33|  6500|        IT|
|   Tina| 35|  7000|   Finance|
+-------+---+------+----------+
only showing top 20 rows



In [3]:
df.createOrReplaceTempView("employees")

In [4]:
result = spark.sql("SELECT department, AVG(salary) as avg_salary from employees group by department")
result.show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|     Sales|5909.090909090909|
|        HR|5490.909090909091|
|   Finance|6646.153846153846|
|        IT|6507.142857142857|
+----------+-----------------+



In [6]:
from pyspark.sql.functions import avg

df.groupby("department").agg(avg("salary").alias("avg_salary")).show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|     Sales|5909.090909090909|
|        HR|5490.909090909091|
|   Finance|6646.153846153846|
|        IT|6507.142857142857|
+----------+-----------------+



1. Mention the details of department and their average salary by taking into consideration all the employees whose age > 25 and the avg salary > 6000 within department

2. Find all the pairs of employees who have same age but different names and display their names as name_a and name_b

3. Use rank() function to assign a rank to employees based on their salary within each department, with the highest salary getting first rank

In [7]:
spark.sql("SELECT department, avg(salary) from employees where age > 25 group by department having avg(salary) > 6000").show()

+----------+-----------------+
|department|      avg(salary)|
+----------+-----------------+
|   Finance|6646.153846153846|
|        IT|6507.142857142857|
+----------+-----------------+



In [8]:
from pyspark.sql.functions import col, avg
df.filter(col("age") > 25).groupby("department").agg(avg("salary").alias("avg_salary")).filter("avg_salary > 6000").show()

+----------+-----------------+
|department|       avg_salary|
+----------+-----------------+
|   Finance|6646.153846153846|
|        IT|6507.142857142857|
+----------+-----------------+



In [9]:
# 2.

spark.sql("select a.name as name_a , b.name as name_b from employees a inner join employees b on a.age=b.age and a.name != b.name").show()

+-------+-------+
| name_a| name_b|
+-------+-------+
|    Bob| Walter|
|    Bob|  Kevin|
|    Bob|   Yara|
|Charlie|   Sean|
|Charlie| George|
|Charlie|   Tina|
|  David|  Mason|
|  David|  Aaron|
|  David|   Rose|
|    Eve|Ulysses|
|    Eve|  Isaac|
|    Eve| Victor|
|  Grace| Rachel|
|  Grace|  Fiona|
|  Grace| Xavier|
|  Grace|   Noah|
|   Hank| Quincy|
|   Hank|  Ethan|
|   Hank|    Sam|
|    Ivy|  Paula|
+-------+-------+
only showing top 20 rows



In [21]:
df_a = df.alias("a")
df_b = df.alias("b")
df_a.join(df_b, (df_a.age == df_b.age) & (df_a.name != df_b.name), "inner").select(col("a.name").alias("name_a"), col("b.name").alias("name_b")).show()

+-------+-------+
| name_a| name_b|
+-------+-------+
|    Bob| Walter|
|    Bob|  Kevin|
|    Bob|   Yara|
|Charlie|   Sean|
|Charlie| George|
|Charlie|   Tina|
|  David|  Mason|
|  David|  Aaron|
|  David|   Rose|
|    Eve|Ulysses|
|    Eve|  Isaac|
|    Eve| Victor|
|  Grace| Rachel|
|  Grace|  Fiona|
|  Grace| Xavier|
|  Grace|   Noah|
|   Hank| Quincy|
|   Hank|  Ethan|
|   Hank|    Sam|
|    Ivy|  Paula|
+-------+-------+
only showing top 20 rows



In [23]:
spark.sql("Select name, salary, department, rank() over (partition by department order by salary desc) as rank from employees").show()

+-------+------+----------+----+
|   name|salary|department|rank|
+-------+------+----------+----+
|  Frank|  7500|   Finance|   1|
|  Wendy|  7400|   Finance|   2|
|  Quinn|  7300|   Finance|   3|
|   Jack|  7200|   Finance|   4|
|Charlie|  7000|   Finance|   5|
|   Tina|  7000|   Finance|   5|
|    Mia|  6800|   Finance|   7|
|  Ethan|  6400|   Finance|   8|
| Quincy|  6400|   Finance|   8|
|  Isaac|  6200|   Finance|  10|
|Ulysses|  6200|   Finance|  10|
|  Aaron|  5500|   Finance|  12|
|  Mason|  5500|   Finance|  12|
|  Laura|  6800|        HR|   1|
|   Yara|  6000|        HR|   2|
|  Peter|  5600|        HR|   3|
|  Daisy|  5600|        HR|   3|
|  Paula|  5600|        HR|   3|
|  Grace|  5300|        HR|   6|
|    Uma|  5200|        HR|   7|
+-------+------+----------+----+
only showing top 20 rows



In [24]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

window_object = Window.partitionBy('department').orderBy(desc('salary'))
df.withColumn("salary_rank", rank().over(window_object)).show()

+-------+---+------+----------+-----------+
|   name|age|salary|department|salary_rank|
+-------+---+------+----------+-----------+
|  Frank| 40|  7500|   Finance|          1|
|  Wendy| 39|  7400|   Finance|          2|
|  Quinn| 37|  7300|   Finance|          3|
|   Jack| 38|  7200|   Finance|          4|
|Charlie| 35|  7000|   Finance|          5|
|   Tina| 35|  7000|   Finance|          5|
|    Mia| 34|  6800|   Finance|          7|
|  Ethan| 33|  6400|   Finance|          8|
| Quincy| 33|  6400|   Finance|          8|
|  Isaac| 32|  6200|   Finance|         10|
|Ulysses| 32|  6200|   Finance|         10|
|  Aaron| 28|  5500|   Finance|         12|
|  Mason| 28|  5500|   Finance|         12|
|  Laura| 34|  6800|        HR|          1|
|   Yara| 30|  6000|        HR|          2|
|  Peter| 29|  5600|        HR|          3|
|  Daisy| 29|  5600|        HR|          3|
|  Paula| 29|  5600|        HR|          3|
|  Grace| 27|  5300|        HR|          6|
|    Uma| 26|  5200|        HR| 